# Desafio 3: Correlação entre Frentes e Votações

## Objetivo
Verificar se deputados de uma mesma frente parlamentar apresentam maior alinhamento
ideológico do que deputados do mesmo partido.

## Estratégia (adaptada devido à instabilidade da API de votações)
Utilizei a classificação ideológica das frentes (Conservadora/Progressista) e a
sobreposição de frentes entre deputados como proxy de alinhamento.

## Método
1. Para cada frente, calcular a proporção de deputados classificados como Progressista
   e Conservador com base nas frentes que participam (usando `ouro.sobreposicao_ideologica`).
2. Para cada partido, calcular o desvio padrão da orientação ideológica dos seus deputados.
3. Comparar: frentes têm desvio menor que partidos? Se sim, frentes são mais coesas
   ideologicamente que partidos.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ouro")
spark.sql("SELECT * FROM ouro.frentes_membros_gold").createOrReplaceTempView("ouro_frentes_view")
spark.sql("SELECT * FROM ouro.sobreposicao_ideologica").createOrReplaceTempView("ouro_sobreposicao_view")
print("Views criadas.")

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.coesao_frentes
USING DELTA
COMMENT 'Camada Ouro - Coesao ideologica dentro de cada frente'
AS
WITH orientacao_por_deputado AS (
    -- Para cada deputado, contar quantas frentes progressistas e conservadoras ele tem
    SELECT 
        id_deputado,
        nome_deputado,
        sigla_partido,
        COUNT(DISTINCT frente_progressista) AS qtd_progressistas,
        COUNT(DISTINCT frente_conservadora) AS qtd_conservadoras
    FROM ouro_sobreposicao_view
    GROUP BY id_deputado, nome_deputado, sigla_partido
),
score_deputado AS (
    -- Criar score: -1 = totalmente conservador, +1 = totalmente progressista
    SELECT 
        *,
        CASE 
            WHEN (qtd_progressistas + qtd_conservadoras) = 0 THEN 0
            ELSE (qtd_progressistas - qtd_conservadoras) * 1.0 / (qtd_progressistas + qtd_conservadoras)
        END AS score_ideologico
    FROM orientacao_por_deputado
),
coesao_frente AS (
    -- Para cada frente, calcular o score medio e o desvio padrao dos membros
    SELECT 
        f.id_frente,
        f.nome_frente,
        f.partido_coordenador,
        COUNT(DISTINCT sd.id_deputado) AS qtd_membros,
        ROUND(AVG(sd.score_ideologico), 4) AS score_medio,
        ROUND(STDDEV(sd.score_ideologico), 4) AS desvio_padrao
    FROM ouro_frentes_view f
    JOIN score_deputado sd ON f.id_deputado = sd.id_deputado
    GROUP BY f.id_frente, f.nome_frente, f.partido_coordenador
    HAVING COUNT(DISTINCT sd.id_deputado) >= 5
)
SELECT * FROM coesao_frente
ORDER BY desvio_padrao ASC

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.coesao_partidos
USING DELTA
COMMENT 'Camada Ouro - Coesao ideologica dentro de cada partido'
AS
WITH orientacao_por_deputado AS (
    SELECT 
        id_deputado,
        nome_deputado,
        sigla_partido,
        COUNT(DISTINCT frente_progressista) AS qtd_progressistas,
        COUNT(DISTINCT frente_conservadora) AS qtd_conservadoras
    FROM ouro_sobreposicao_view
    GROUP BY id_deputado, nome_deputado, sigla_partido
),
score_deputado AS (
    SELECT 
        *,
        CASE 
            WHEN (qtd_progressistas + qtd_conservadoras) = 0 THEN 0
            ELSE (qtd_progressistas - qtd_conservadoras) * 1.0 / (qtd_progressistas + qtd_conservadoras)
        END AS score_ideologico
    FROM orientacao_por_deputado
)
SELECT 
    sigla_partido,
    COUNT(DISTINCT id_deputado) AS qtd_deputados,
    ROUND(AVG(score_ideologico), 4) AS score_medio,
    ROUND(STDDEV(score_ideologico), 4) AS desvio_padrao
FROM score_deputado
WHERE sigla_partido IS NOT NULL
GROUP BY sigla_partido
HAVING COUNT(DISTINCT id_deputado) >= 5
ORDER BY desvio_padrao ASC

In [0]:
%sql
-- Comparativo: Frentes vs Partidos
-- Media de desvio padrao (quanto menor, mais coeso)
SELECT 
    'Frentes' AS tipo_agrupamento,
    ROUND(AVG(desvio_padrao), 4) AS media_desvio,
    ROUND(MIN(desvio_padrao), 4) AS menor_desvio,
    ROUND(MAX(desvio_padrao), 4) AS maior_desvio,
    COUNT(*) AS qtd_grupos
FROM ouro.coesao_frentes
UNION ALL
SELECT 
    'Partidos' AS tipo_agrupamento,
    ROUND(AVG(desvio_padrao), 4) AS media_desvio,
    ROUND(MIN(desvio_padrao), 4) AS menor_desvio,
    ROUND(MAX(desvio_padrao), 4) AS maior_desvio,
    COUNT(*) AS qtd_grupos
FROM ouro.coesao_partidos


In [0]:
%sql
SELECT 
    nome_frente,
    partido_coordenador,
    qtd_membros,
    score_medio,
    desvio_padrao
FROM ouro.coesao_frentes
ORDER BY desvio_padrao ASC
LIMIT 10

In [0]:
%sql
SELECT 
    sigla_partido,
    qtd_deputados,
    score_medio,
    desvio_padrao
FROM ouro.coesao_partidos
ORDER BY desvio_padrao ASC
LIMIT 10

# Conclusão do Desafio 3

## Resultado com dados reais de voto (atualizado em 05/05/2026)
Obtivemos o CSV oficial de votos de 2024 do portal de Dados Abertos:
https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-2024.csv

Com 116.400 votos de 572 deputados em 448 votações, o resultado é:

| Grupo | Similaridade Média |
|-------|-------------------|
| Partidos | 84,3% |
| Frentes | 69,5% |

Deputados do mesmo PARTIDO votam 84,3% igual.
Deputados da mesma FRENTE votam apenas 69,5% igual.
Diferença de 14,8 pontos percentuais.

## Resultado anterior (proxy ideológico)
Antes dos dados reais, usamos classificação de frentes como proxy.
O resultado foi: Partidos 0,324 vs Frentes 0,338 (desvio padrão).
Ambos os métodos apontam na MESMA direção: partidos são mais coesos.

## Interpretação
A disciplina partidária é o fator dominante no voto parlamentar,
muito mais forte que a afinidade temática das frentes.

In [0]:
%sql
-- Comparativo final com dados reais de voto
-- Tabelas criadas no notebook 14_cdc_proposicoes
SELECT 
    'Frentes' AS tipo_grupo,
    ROUND(AVG(similaridade_pct), 1) AS similaridade_media,
    ROUND(MIN(similaridade_pct), 1) AS min_similaridade,
    ROUND(MAX(similaridade_pct), 1) AS max_similaridade,
    COUNT(*) AS qtd_grupos
FROM ouro.similaridade_frentes
UNION ALL
SELECT 
    'Partidos' AS tipo_grupo,
    ROUND(AVG(similaridade_pct), 1) AS similaridade_media,
    ROUND(MIN(similaridade_pct), 1) AS min_similaridade,
    ROUND(MAX(similaridade_pct), 1) AS max_similaridade,
    COUNT(*) AS qtd_grupos
FROM ouro.similaridade_partidos